# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 01.03 · Limpieza y troceado incremental

Recompone la vista canónica desde los checkpoints sincronizados, recupera VTT locales utilizables y crea chunks deterministas únicamente para transcripciones nuevas o modificadas.

La normalización NFKC aplicada al texto sigue las formas de normalización Unicode [1], y las huellas de transcripción, texto e identificadores estables usan SHA-256 [2]. La longitud, los límites de caracteres, el solapamiento y las reglas de deduplicación son parámetros locales versionados. Cada firma tiene un archivo recuperable: cambiarla mueve los derivados vigentes y volver a una firma restaura sus bytes verificados.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [1]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')


raíz,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4
backend,local


## Preparación reproducible y consolidación local

In [2]:
from moderacion_peru.acquisition import consolidate_available_transcripts, materialize_transcripts_by_channel
from moderacion_peru.chunk_optimization import activate_chunking_configuration, load_chunking_configuration
from moderacion_peru.incremental import materialize_chunk_records
from moderacion_peru.io import read_jsonl
SOURCE=ROOT/'datos/raw/transcripts_raw.jsonl'
TRANSCRIPTS_BY_CHANNEL=ROOT/'datos/raw/transcripts_by_channel'
TRANSCRIPTS_CACHE=ROOT/'datos/raw/transcripts_cache'
VTT_BY_VIDEO=ROOT/'datos/raw/vtt_by_video'
OUTPUT=ROOT/'datos/processed/chunks_v2.jsonl'
VERSION_INDEX=ROOT/'datos/processed/chunking_v2_versions.jsonl'
MATERIALIZATION_MANIFEST=ROOT/'datos/processed/chunk_materialization_manifest.json'
CHUNK_CONFIG_PATH=ROOT/'config/chunking.json'
REBUILD_CHUNKS_FROM_ZERO=False  # True: copia recuperable y reconstrucción total; después vuelva a False
CHUNK_CONFIG=load_chunking_configuration(CHUNK_CONFIG_PATH)
activation=activate_chunking_configuration(ROOT,CHUNK_CONFIG,source='01_03_materialization')
consolidation=consolidate_available_transcripts(ROOT,SOURCE,cache_dir=TRANSCRIPTS_CACHE,channel_dir=TRANSCRIPTS_BY_CHANNEL,vtt_dir=VTT_BY_VIDEO if VTT_BY_VIDEO.is_dir() else None)
channel_checkpoint=materialize_transcripts_by_channel(SOURCE,TRANSCRIPTS_BY_CHANNEL)
show_result('Estado de la configuración de chunks',activation,tone='success')
show_summary('Cobertura consolidada antes del troceado',{'videos_canónicos':consolidation['canonical_videos'],'VTT_recuperados':consolidation['vtt_added'],'VTT_demasiado_cortos':consolidation['vtt_recovery']['too_short'],'partes_por_canal':channel_checkpoint['total_channel_files'],'canales':channel_checkpoint['total_channels'],'canónico_local':SOURCE,'checkpoint_Git':TRANSCRIPTS_BY_CHANNEL},tone='success')
if consolidation['vtt_recovery']['too_short_records']:
    show_table('VTT excluidos por menos de 200 caracteres',consolidation['vtt_recovery']['too_short_records'],max_rows=len(consolidation['vtt_recovery']['too_short_records']))

schema_version,1.0
status,already_active_noop
signature,cb4f8eefae6a2ee1baeca642cf20703ddd9a55684444cafd687f8543f08ac111
configuration,"Ver detalle{ ""max_seconds"": 30.0, ""max_characters"": 600, ""min_characters"": 90, ""overlap_words"": 12 }"
source,01_03_materialization


videos_canónicos,5002
VTT_recuperados,0
VTT_demasiado_cortos,7
partes_por_canal,339
canales,336
canónico_local,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/datos/raw/transcripts_raw.jsonl
checkpoint_Git,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/datos/raw/transcripts_by_channel


video_id,characters,selected_vtt
4Qm_-mYJHt8,35,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/datos/raw/vtt_by_video/4Qm_-mYJHt8.es.vtt
E9l11PsuHCw,96,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/datos/raw/vtt_by_video/E9l11PsuHCw.es.vtt
KTfK2FniMLs,105,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/datos/raw/vtt_by_video/KTfK2FniMLs.es.vtt
SDAMs7BqLVs,80,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/datos/raw/vtt_by_video/SDAMs7BqLVs.es.vtt
T2hFz8Z3LmQ,116,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/datos/raw/vtt_by_video/T2hFz8Z3LmQ.es.vtt
fx8tGunSflw,14,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/datos/raw/vtt_by_video/fx8tGunSflw.es.vtt
m2ZL6kvYA3I,82,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/datos/raw/vtt_by_video/m2ZL6kvYA3I.es.vtt


## Materialización

In [3]:
from tqdm.auto import tqdm
transcript_total=consolidation['canonical_videos']
chunk_progress=tqdm(total=transcript_total,desc='Materializando transcripciones',unit='video')
def report_chunk_progress(event):
    chunk_progress.update(event.get('advance',1))
    chunk_progress.set_postfix(nuevos=event.get('new_or_changed_videos',0),sin_cambios=event.get('unchanged_videos',0),chunks_video=event.get('generated_chunks_for_video',0))
try:
    materialization=materialize_chunk_records(ROOT,read_jsonl(SOURCE),source_path=SOURCE,output_path=OUTPUT,version_index_path=VERSION_INDEX,manifest_path=MATERIALIZATION_MANIFEST,rebuild=REBUILD_CHUNKS_FROM_ZERO,progress_callback=report_chunk_progress,**CHUNK_CONFIG)
finally:
    chunk_progress.close()
show_result('Resultado de limpieza y troceado',materialization['stats'],tone='success')
show_summary('Cobertura final reportable',{'transcripciones':materialization['coverage']['transcript_videos'],'videos_con_chunks':materialization['coverage']['videos_with_chunks'],'videos_sin_chunks':materialization['coverage']['videos_without_chunks'],'chunks_totales':materialization['outputs']['chunks']['rows'],'reconstrucción_total':REBUILD_CHUNKS_FROM_ZERO,'respaldo':materialization['backup'],'manifiesto':MATERIALIZATION_MANIFEST},tone='warning' if materialization['coverage']['videos_without_chunks'] else 'success')
if materialization['coverage']['video_ids_without_chunks']:
    show_table('Videos evaluados sin chunks materializables',[{'video_id':video_id} for video_id in materialization['coverage']['video_ids_without_chunks']],max_rows=len(materialization['coverage']['video_ids_without_chunks']))

Materializando transcripciones:   0%|          | 0/5002 [00:00<?, ?video/s]

transcripts_seen,5002
unchanged_videos,5002
new_or_changed_videos,0
generated_chunks,0
new_unique_chunks,0
videos_with_generated_chunks,0
videos_with_new_unique_chunks,0
videos_without_generated_chunks,0
videos_without_new_unique_chunks,0
added,0
duplicate_ids,0


transcripciones,5002
videos_con_chunks,4992
videos_sin_chunks,10
chunks_totales,166940
reconstrucción_total,No
respaldo,archivo/chunk_rebuilds/20260807T165807714288Z
manifiesto,D:/trabajo_PLN/Trabajo_PLN-MIA-Grupo4/datos/processed/chunk_materialization_manifest.json


video_id
0gZaIyxBANk
1MPPLtM6BA4
8QGI_WoaN-I
NGBMshQT35I
ObvZFla0Zq4
iDHUBY-MOL4
sOiJB14im-g
su8i8FN0V-8
yEgGOJ37VsY
yO-RcPkIpDc


## Referencias

[1] Unicode Consortium, "Unicode Normalization Forms," Unicode Standard Annex No. 15, rev. 57, Jul. 2025. [Online]. Available: https://www.unicode.org/reports/tr15/tr15-57.html

[2] National Institute of Standards and Technology, "Secure Hash Standard (SHS)," FIPS PUB 180-4, Aug. 2015, doi: 10.6028/NIST.FIPS.180-4.